In [ ]:
# Colab setup: install packages not preinstalled on Colab (safe to re-run)
!pip install -q abess

# Reproduce-then-Extend — Advance-Care-Planning Engagement (Health Services Research) · **Day 1 tutorial**

> **Published study.** Han, Z., Su, J. & Ma, G. (2025). "Factors influencing the participation of surrogate
> decision-makers for advance care planning." *PLOS ONE* 20 (doi:10.1371/journal.pone.0325551; data in the
> article's S1 Dataset). A **multiple linear regression** of surrogate decision-makers' **ACP-engagement score**
> on personal and clinical factors (**n = 276**).

## Background

When a seriously ill patient cannot decide for themselves, a **surrogate** decides — and whether that surrogate
has engaged in **advance care planning (ACP)** shapes end-of-life care. Han, Su & Ma survey 276 surrogate
decision-makers of advanced-cancer patients and model their **ACP engagement** (a validated 17-item score,
range 17–85) on personal, family, and clinical characteristics.

## Data and codebook

**Unit of analysis:** a surrogate decision-maker; n = 276. Outcome `ACP_engagement` is the ACP-17-SDM
engagement score (17–85). Predictors include `decision_experience` (prior medical-decision experience),
`education`, `ACP_knowledge`, `treatment_expenditure`, income, family size, relationship to the patient, the
patient's cancer type and self-care ability, and readiness measures.

## Descriptive results

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.linear_model import LassoCV, RidgeCV, ElasticNetCV, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

In [ ]:
d = pd.read_csv('https://raw.githubusercontent.com/desmarais-lab/desmarais-lab.github.io/master/istanbul_bilgi_ml_files/data/surrogate_acp.csv')
outcome = 'ACP_engagement'
preds = [c for c in d.columns if c != outcome]
print(f'{d.shape[0]} surrogates x {len(preds)} predictors')
d[[outcome]].describe().T[['mean','std','min','max']].round(2)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9, 3.2))
ax[0].hist(d[outcome], bins=15, color='#a6cee3', edgecolor='white'); ax[0].set_title('Outcome: ACP engagement (17-85)')
cors = d[preds].apply(lambda c: pd.to_numeric(c,errors='coerce').corr(d[outcome])).sort_values()
ax[1].barh([t[:20] for t in cors.index], cors.values, color=['#1f78b4' if v>0 else '#e31a1c' for v in cors.values])
ax[1].axvline(0, color='grey'); ax[1].set_title('Correlation with engagement'); plt.tight_layout()

## Reproduce the published regression

The paper's model regresses ACP engagement on the surrogate/clinical factors.

In [ ]:
X = d[preds].apply(pd.to_numeric, errors='coerce'); y = pd.to_numeric(d[outcome], errors='coerce')
ok = X.notna().all(1) & y.notna()
ols = sm.OLS(y[ok], sm.add_constant(X[ok])).fit()
print(f'n = {int(ols.nobs)}, R^2 = {ols.rsquared:.3f} (adj. {ols.rsquared_adj:.3f})')
for v in ['decision_experience','education','ACP_knowledge','treatment_expenditure']:
    if v in preds:
        star = '***' if ols.pvalues[v]<0.001 else '**' if ols.pvalues[v]<0.01 else '*' if ols.pvalues[v]<0.05 else ''
        print(f'   {v:22s} beta = {ols.params[v]:+.3f}  p = {ols.pvalues[v]:.3f} {star}')

**Confirmation against the published study.** The regression recovers the paper's headline finding: prior
**medical-decision experience**, **education**, **ACP knowledge**, and **treatment expenditure** are all
**positive, significant** predictors of ACP engagement — the strong positive associations Han, Su & Ma report.
(Their full model, adding three derived psychological scales we don't reconstruct here, reaches adjusted
$R^2 \approx 0.57$; the factors above reproduce with the same signs and significance.)

## Regularization & out-of-sample prediction

With `r len(preds)` predictors on 276 surrogates, we compare OLS to penalized regression out of sample.

In [ ]:
Xv, yv = X[ok].values, y[ok].values
res = {k: [] for k in ['OLS','Lasso','Ridge','ElasticNet']}
for r in range(50):
    Xtr,Xte,ytr,yte = train_test_split(Xv, yv, test_size=0.30, random_state=r)
    sc = StandardScaler().fit(Xtr); Ztr, Zte = sc.transform(Xtr), sc.transform(Xte)
    res['OLS'].append(r2_score(yte, LinearRegression().fit(Ztr,ytr).predict(Zte)))
    res['Lasso'].append(r2_score(yte, LassoCV(cv=5,random_state=0,max_iter=100000).fit(Ztr,ytr).predict(Zte)))
    res['Ridge'].append(r2_score(yte, RidgeCV(alphas=np.logspace(-2,3,40)).fit(Ztr,ytr).predict(Zte)))
    res['ElasticNet'].append(r2_score(yte, ElasticNetCV(cv=5,l1_ratio=0.5,random_state=0,max_iter=100000).fit(Ztr,ytr).predict(Zte)))
for k,v in res.items(): print(f'{k:12s} held-out R^2 = {np.mean(v):+.3f}')

The engagement model predicts new surrogates out of sample, and the lasso keeps the compact set of drivers
— prior decision experience, education, and ACP knowledge — that the paper highlights.

## Takeaway

A health-services study reproduced faithfully: the regression recovers Han, Su & Ma's reported drivers of
advance-care-planning engagement among surrogate decision-makers, then extends the model with penalized
regression evaluated out of sample.

## Recommended exercises

1. Add interaction terms (e.g. decision experience × ACP knowledge) and check the held-out gain.
2. Compare lasso-selected drivers here with the paper's seven reported predictors.
3. Predict the `readiness` subscale instead and see which factors matter.